# Agent Evaluation

For RAG, we used the A->Q-A' setup where: 

- A = Orignal answer in the FAQ's 
- Q = The question generated from that answer
- A' = Answer produced by our RAG system

We use the same setup for Agents BUT A' comes from tan agent instead of the fixed RAG pipeline. 

Additionally, we save the trajectory, (i.e., tool calling) so as inspect what the agent did before producing the final answer. 

## Loading the data 

We use the same ground truth data as before

In [1]:
import pandas as pd 

df_ground_truth = pd.read_csv('data/ground_truth-new.csv')

ground_truth = df_ground_truth.to_dict(orient="records")

#### Load the FAQ docs and the search index

In [2]:
from utils.ingest import load_faq_data, build_index 

documents = load_faq_data()

documents_llm = [doc for doc in documents if doc["course"]=="llm-zoomcamp"]

index = build_index(documents)

#### Create the Lookup Table

In [3]:
doc_idx = {doc["id"]: doc for doc in documents_llm}

### Running the Agent

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient


load_dotenv()

openai_client = OpenAI()

#### Define the search tool

In [5]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ dataase for entries matching the given query.
    """

    results = index.search(
        query, 
        num_results=5, 
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={'courese': "llm-zommcamp"}
    )

    return results 

#### Create the runner

In [6]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [7]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [13]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Can I still join the course if I found it late?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"join the course late late enrollment found it late can I still join"}', call_id='call_c9ET6Sh1gUKfTxw3X5yOjtLC', name='search', type='function_call', id='fc_0a60cac36fddc93d006a565294849481918e32b8dac6730bbd', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_c9ET6Sh1gUKfTxw3X5yOjtLC',
  'output': '{\n  "error": "FilterValidationError: Unknown filter field \'courese\'. Valid fields are: [\'course\']"\n}'},
 ResponseFunctionToolCall(arguments='{"query":"late enrollment join course found it late"}', call_id='call_N93ZbPEGt0Lb8vUOyHKCyfJy', name='search', type='function_call', id='fc_0a60cac

In [14]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls


tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"join the course late late enrollment found it late can I still join"}'},
 {'name': 'search',
  'arguments': '{"query":"late enrollment join course found it late"}'}]

In [15]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [17]:
print(doc_id)
print(original_doc)
print(answer_orig)

74eb249bbf
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [16]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Can I still join the course if I found it late?',
 'answer_agent': 'I’m sorry, but I couldn’t retrieve the FAQ result for this question due to a search/filter issue in the course database.\n\nIf you want, I can help you draft a short message to the course team asking whether late enrollment is still possible.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"join the course late late enrollment found it late can I still join"}'},
  {'name': 'search',
   'arguments': '{"query":"late enrollment join course found it late"}'}],
 'cost': Decimal('0.00082575'),
 'document': '74eb249bbf'}

### Do it for multiple questions

Wrap everything above in a function 

In [19]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [21]:
# Run it on a small sample

from concurrent.futures import ThreadPoolExecutor
from utils.evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

100%|██████████| 50/50 [00:52<00:00,  1.05s/it]


In [22]:
df_agent = pd.DataFrame(agent_answers)
df_agent["cost"].sum()

Decimal('0.04875450')

In [23]:
df_agent.to_csv("data/agent-answers.csv", index=False)

The point of the agent is not to use the tools; it is to answer the question using the correct tools the fewest number of times. 

As such, we need to define our scores to evaluate our agent properly. 

In [24]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [25]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

### Define the Judge function

In [27]:
import json
from utils.evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [28]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent did not provide the FAQ answer. The ground truth says you can still join the course, with the caveat that to receive a certificate you must submit the project while submissions are still being accepted. Instead, the agent said it could not retrieve the FAQ entry and offered to retry. This misses the key information entirely, so the answer is incorrect.', answer_score='bad', trajectory_reasoning='The search queries were relevant to the question and included important keywords like late, join, course, and deadline. However, the agent made two very similar searches that were mostly duplicate and did not meaningfully refine the query. The tool use was reasonable in number, but it failed to support the final answer because the agent gave up without using the likely relevant FAQ content.', trajectory_score='bad')

## Run the Agent Judge over everything

In [29]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [30]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

100%|██████████| 50/50 [00:20<00:00,  2.45it/s]


In [31]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [32]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [33]:
calc_total_price(usages)

0.05971724999999999

In [34]:
# Check the Scores 
df_agent_eval["answer_score"].value_counts(normalize=True)

answer_score
bad     0.96
good    0.04
Name: proportion, dtype: float64

In [35]:
df_agent_eval["trajectory_score"].value_counts(normalize=True)

trajectory_score
good    0.54
bad     0.46
Name: proportion, dtype: float64